# Contradiction-Aware Legal Graph RAG
## Complete Research Pipeline

```
Real Cases (HuggingFace)      Synthetic Cases (LLM)
        ↓                              ↓
   [data_processor.py — CASE-LEVEL SPLIT]
        ↓          ↓          ↓
      Train       Val       Test (GOLD)
        ↓
[bert_finetuner.py] → Fine-Tuned BERT
        ↓
[knowledge_graph.py] → Graph from BERT predictions
        ↓
[evaluation_pipeline.py] → 6 ablations + Blind LLM Judge
```
⚠️ **Runtime → T4 GPU** (required for BERT fine-tuning)

In [ ]:
# CELL 1: Install dependencies
!pip install -q transformers torch scikit-learn datasets
!pip install -q sentence-transformers spacy pandas matplotlib networkx pyvis
!python -m spacy download en_core_web_sm -q
print('Done')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2: API KEYS — set ONE or BOTH providers here
# ═══════════════════════════════════════════════════

# OPTION A: Google Gemini (free: 1500 req/day fast, 500/day smart)

GOOGLE_API_KEY = ''

# OPTION B: OpenRouter  (free: 200 req/day, auto-selects best model)

OPENROUTER_API_KEY = ''

# ── Which provider to use ──────────────────────────────────────
# 'auto'        → OpenRouter if set, else Gemini, else mock
# 'gemini'      → force Gemini
# 'openrouter'  → force OpenRouter  ← switch here when Gemini quota exhausted
LLM_PROVIDER = 'auto'

# ── OpenRouter model (ignored if using Gemini) ─────────────────

OR_MODEL = 'openrouter/free'

# ═══════════════════════════════════════════════════
import os, sys
if GOOGLE_API_KEY:    os.environ['GOOGLE_API_KEY']    = GOOGLE_API_KEY
if OPENROUTER_API_KEY: os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY

OUTPUT_PATH = '/content/drive/MyDrive/nlp/results'
MAX_CASES   = 200
MAX_SCAN    = 200000

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUTPUT_PATH, exist_ok=True)
sys.path.insert(0, '.')

# ── Initialize unified LLM client (used by ALL cells below) ────
from llm_client import get_client
llm = get_client(
    provider=LLM_PROVIDER,
    gemini_key=GOOGLE_API_KEY or None,
    openrouter_key=OPENROUTER_API_KEY or None,
    openrouter_model=OR_MODEL,
)
print(f'\nReady. Drive: {OUTPUT_PATH}')
print(f'LLM provider : {llm.provider}')
print(f'Configured   : {llm.is_configured}')
if not llm.is_configured:
    print('   No key set — LLM steps will run in mock mode')
    print('  Data collection (HuggingFace) and BERT training still work.')

## Step 1: Collect Real Cases (no API key needed)
Streams `common-pile/caselaw_access_project` from HuggingFace.
Finds cases with majority + dissent opinions — guaranteed contradictions.

In [ ]:
# CELL 3: Collect dissent cases — SAFE TO RESTART (checkpoint)
from hf_dissent_fetcher import HFFetchConfig, fetch_dissent_cases

config = HFFetchConfig(
    min_quality_score=40, max_cases=MAX_CASES, max_scan=MAX_SCAN,
    checkpoint_every=5, checkpoint_path='hf_checkpoint.json',
    output_path='dissent_cases.json', drive_output_path=OUTPUT_PATH,
)
cases = fetch_dissent_cases(config, verbose=True)
print(f'Real dissent cases: {len(cases)}')
for c in cases[:3]:
    print(f'  {c.case_id[:35]} | {c.crime_type} | score={c.quality_score}')

## Step 2: Generate Synthetic Cases
10 structured cases with explicit prosecution/defense/court claims.
Uses your configured LLM provider (Gemini or OpenRouter).

In [ ]:
# CELL 4: Synthetic augmentation via configured LLM
import shutil
from synthetic_augmenter import generate_synthetic_cases

syn_cases = generate_synthetic_cases(
    llm=llm,                          # ← uses provider from Cell 2
    output_path='synthetic_cases.json',
    drive_path=OUTPUT_PATH,
)
print(f'Synthetic cases: {len(syn_cases)}')
c = syn_cases[0]
print(f'Sample ({c["topic"]}): {c["prosecution_claims"][0][:80]}...')

## Step 3: Case-Level Split
All pairs from one case go to ONE split — prevents data leakage.
- Train: real + synthetic | Val/Test: real only (GOLD SET)

In [ ]:
# CELL 5: Case-level split + pair generation
# Human pairs added to TRAIN only — val/test stay clean
from data_processor import DataProcessor
import importlib, data_processor
importlib.reload(data_processor)
from data_processor import DataProcessor
import shutil, os

proc = DataProcessor(
    google_api_key=None,
    bert_model_path=None,
    max_pairs_per_case=30,
    seed=42,
)
splits = proc.process(
    dissent_cases_json='dissent_cases.json',
    synthetic_cases_json='synthetic_cases.json',
    human_pairs_json='human_pairs.json',  # ← added to train
    output_dir='.', drive_path=OUTPUT_PATH,
)
train_df, val_df, test_df = splits['train'], splits['val'], splits['test']

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Train label_source breakdown:')
print(train_df['label_source'].value_counts().to_string())
print(f'\nTrain balance: {train_df["label"].mean():.1%} positive')
print('Test is GOLD SET — not used until evaluation')

## Step 4: EDA

In [ ]:
# CELL 6: Exploratory Data Analysis
import pandas as pd, matplotlib.pyplot as plt, shutil

df_all = pd.concat([train_df, val_df, test_df])
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

df_all['label'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['#3498db','#e74c3c'], edgecolor='white')
axes[0].set_title('Label Distribution')
axes[0].set_xticklabels(['Not Contradiction','Contradiction'], rotation=15)

if 'topic' in df_all.columns:
    df_all['topic'].value_counts()[:8].plot(kind='barh', ax=axes[1], color='#9b59b6')
    axes[1].set_title('Topic Distribution')

df_all['text_a'].str.len().hist(bins=30, ax=axes[2], color='#2ecc71', alpha=0.8)
axes[2].set_title('Claim Length (chars)')

plt.suptitle('EDA — Legal Contradiction Dataset', fontweight='bold')
plt.tight_layout()
plt.savefig('eda.png', dpi=150, bbox_inches='tight')
shutil.copy('eda.png', f'{OUTPUT_PATH}/eda.png')
plt.show()
print(f'Total: {len(df_all)} | pos={df_all["label"].sum()} | neg={(df_all["label"]==0).sum()}')
print(f'Sources: {df_all["label_source"].value_counts().to_dict()}')

## Step 5: Fine-Tune BERT


In [ ]:
# CELL 7: Fine-tune BERT
import torch, shutil
if not torch.cuda.is_available():
    raise SystemExit('GPU required: Runtime → Change runtime type → T4')
print(f'GPU: {torch.cuda.get_device_name(0)}')

from bert_finetuner import BERTFineTuner, plot_training_history
trainer = BERTFineTuner(model_name='bert-base-uncased', output_dir='bert_model')
history = trainer.train(train_df, val_df, epochs=4, batch_size=16, lr=2e-5)
trainer.save()
plot_training_history(history, 'training_history.png')
shutil.copy('training_history.png', f'{OUTPUT_PATH}/training_history.png')
print('BERT saved to bert_model/')

## Step 6: Hybrid Validation
BERT confidence filter (p>0.8) + LLM spot-check (~25% sample).

In [ ]:
# CELL 8: Hybrid validation on train set
# BERT is now trained → can filter noisy labels
# LLM spot-check uses configured provider (Gemini or OpenRouter)
from data_processor import hybrid_validation, ClaimPair

train_pairs = [
    ClaimPair(
        pair_id=row.get('pair_id',''), case_id=row.get('case_id',''),
        text_a=row['text_a'], text_b=row['text_b'],
        party_a=row.get('party_a',''), party_b=row.get('party_b',''),
        topic=row.get('topic',''), label=int(row['label']),
        confidence=float(row.get('confidence', 0.9)),
        label_source=row.get('label_source','structural'),
        split='train',
    )
    for _, row in train_df.iterrows()
]

# google_api_key param kept for compat; function now uses get_client() internally
validated_pairs = hybrid_validation(
    train_pairs,
    bert_model_path='bert_model/best_model',
    bert_threshold=0.8,
    llm_sample_rate=0.25,
    split='train',
)
print(f'After validation: {len(validated_pairs)}/{len(train_pairs)} pairs kept')

## Step 7: Build Knowledge Graph
Graph edges come from fine-tuned BERT predictions.

In [ ]:
# CELL 9: Build knowledge graph from BERT predictions
import json, shutil
from knowledge_graph import LegalKnowledgeGraph

with open('dissent_cases.json') as f:   raw_cases = json.load(f)
with open('synthetic_cases.json') as f: syn_raw   = json.load(f)

train_case_ids  = set(train_df['case_id'].unique())
train_cases_data = [
    c for c in raw_cases + syn_raw
    if c.get('case_id','') in train_case_ids
]
print(f'Building graph from {len(train_cases_data)} train cases...')

kg = LegalKnowledgeGraph()
kg.build(
    cases_data=train_cases_data,
    bert_model_path='bert_model/best_model',
    bert_threshold=0.65,
    edge_source='bert',
    add_contradicts=True,
    add_resolved_by=True,
    verbose=True,
)
kg.save('legal_kg.pkl')
shutil.copy('legal_kg.pkl', f'{OUTPUT_PATH}/legal_kg.pkl')

print('\nGraph stats:')
for k, v in kg.stats().items(): print(f'  {k}: {v}')

## Step 8: Ablation Study
6 configurations × blind LLM judge on GOLD test set.
All systems use the same LLM provider configured in Cell 2.

## Step 8: Ablation Study

**One query per cell** — saves checkpoint after each.

How queries work:
- Generated automatically from real BERT-predicted contradictions in the graph
- One query per unique case — so each question is grounded in actual case content
- Falls back to topic templates if no LLM key

Safe to restart: each cell checks checkpoint and skips if already done.

In [ ]:
# CELL 10-SETUP: Build systems + generate queries from graph
# Run once. Reload from checkpoint automatically on restart.
import json, shutil, importlib, os

import graph_rag_pipeline, evaluation_pipeline
importlib.reload(graph_rag_pipeline)
importlib.reload(evaluation_pipeline)
from evaluation_pipeline import (
    evaluate_classification, aggregate_verdicts,
    print_ablation_table, plot_ablation, save_all_results,
    build_rag_documents, generate_queries_from_graph,
)

# Documents from full case texts
documents = build_rag_documents(
    dissent_cases_json='dissent_cases.json',
    synthetic_cases_json='synthetic_cases.json',
    train_case_ids=set(train_df['case_id'].unique()),
)

# Build 6 ablation systems
systems = graph_rag_pipeline.build_ablation_systems(
    kg_full=kg,
    documents=documents,
    cases_data=train_cases_data,
    bert_model_path='bert_model/best_model',
    llm=llm,
)

# Generate 8 queries grounded in real contradictions from the graph
QUERIES_FILE  = 'eval_queries.json'
VERDICTS_FILE = 'judge_verdicts_checkpoint.json'

if os.path.exists(QUERIES_FILE):
    with open(QUERIES_FILE) as f:
        eval_queries = json.load(f)
    print(f'Loaded {len(eval_queries)} existing queries')
else:
    eval_queries = generate_queries_from_graph(kg, n_queries=8, llm=llm)
    with open(QUERIES_FILE, 'w') as f:
        json.dump(eval_queries, f, indent=2)
    shutil.copy(QUERIES_FILE, f'{OUTPUT_PATH}/{QUERIES_FILE}')
    print(f'Generated and saved {len(eval_queries)} queries')

# Load existing verdict checkpoint
if os.path.exists(VERDICTS_FILE):
    with open(VERDICTS_FILE) as f:
        saved_verdicts = json.load(f)
    done_queries = {v['query'] for v in saved_verdicts}
    print(f'Checkpoint: {len(saved_verdicts)} verdicts, {len(done_queries)} queries done')
else:
    saved_verdicts = []
    done_queries   = set()
    print('No checkpoint — starting fresh')

judge = evaluation_pipeline.BlindLLMJudge(llm=llm)

print(f'\nQueries to evaluate:')
for i, q in enumerate(eval_queries):
    status = 'DONE' if q['query'] in done_queries else 'TODO'
    print(f'  Q{i+1} [{status}] ({q["topic"]}) {q["query"][:60]}')

In [ ]:
# CELL 10-Q1: Run judge on query 1/8
# This cell skips automatically if already done

_q = eval_queries[0] if 0 < len(eval_queries) else None
if _q is None:
    print('Query 1 not available — fewer than 1 queries generated')
elif _q['query'] in done_queries:
    print(f'Q1 already done: {_q["query"][:60]} — skipping')
else:
    print(f'Q1: {_q["query"]}')
    print(f'  Case: {{_q["case_id"]}} | Topic: {{_q["topic"]}}')
    if _q['majority_hint']:
        print(f'  Majority: {{_q["majority_hint"][:80]}}...')
    if _q['dissent_hint']:
        print(f'  Dissent:  {{_q["dissent_hint"][:80]}}...')
    print()

    new_vs = judge.judge_all(
        systems=systems,
        queries=[_q['query']],
        verbose=True,
    )

    # Save immediately
    saved_verdicts.extend([vars(v) for v in new_vs])
    done_queries.add(_q['query'])
    with open(VERDICTS_FILE, 'w') as f:
        json.dump(saved_verdicts, f, indent=2)
    shutil.copy(VERDICTS_FILE, f'{OUTPUT_PATH}/{VERDICTS_FILE}')

    print(f'\nQ1 done. Total verdicts: {len(saved_verdicts)}. Queries done: {len(done_queries)}/{len(eval_queries)}')
    for v in sorted(new_vs, key=lambda x: -x.avg_score):
        print(f'  {{v.system_name[:40]:<40}} avg={{v.avg_score:.2f}}')

In [ ]:
# CELL 10-Q2: Run judge on query 2/8
# This cell skips automatically if already done

_q = eval_queries[1] if 1 < len(eval_queries) else None
if _q is None:
    print('Query 2 not available — fewer than 2 queries generated')
elif _q['query'] in done_queries:
    print(f'Q2 already done: {_q["query"][:60]} — skipping')
else:
    print(f'Q2: {_q["query"]}')
    print(f'  Case: {{_q["case_id"]}} | Topic: {{_q["topic"]}}')
    if _q['majority_hint']:
        print(f'  Majority: {{_q["majority_hint"][:80]}}...')
    if _q['dissent_hint']:
        print(f'  Dissent:  {{_q["dissent_hint"][:80]}}...')
    print()

    new_vs = judge.judge_all(
        systems=systems,
        queries=[_q['query']],
        verbose=True,
    )

    # Save immediately
    saved_verdicts.extend([vars(v) for v in new_vs])
    done_queries.add(_q['query'])
    with open(VERDICTS_FILE, 'w') as f:
        json.dump(saved_verdicts, f, indent=2)
    shutil.copy(VERDICTS_FILE, f'{OUTPUT_PATH}/{VERDICTS_FILE}')

    print(f'\nQ2 done. Total verdicts: {len(saved_verdicts)}. Queries done: {len(done_queries)}/{len(eval_queries)}')
    for v in sorted(new_vs, key=lambda x: -x.avg_score):
        print(f'  {{v.system_name[:40]:<40}} avg={{v.avg_score:.2f}}')

In [ ]:
# CELL 10-Q3: Run judge on query 3/8
# This cell skips automatically if already done

_q = eval_queries[2] if 2 < len(eval_queries) else None
if _q is None:
    print('Query 3 not available — fewer than 3 queries generated')
elif _q['query'] in done_queries:
    print(f'Q3 already done: {_q["query"][:60]} — skipping')
else:
    print(f'Q3: {_q["query"]}')
    print(f'  Case: {{_q["case_id"]}} | Topic: {{_q["topic"]}}')
    if _q['majority_hint']:
        print(f'  Majority: {{_q["majority_hint"][:80]}}...')
    if _q['dissent_hint']:
        print(f'  Dissent:  {{_q["dissent_hint"][:80]}}...')
    print()

    new_vs = judge.judge_all(
        systems=systems,
        queries=[_q['query']],
        verbose=True,
    )

    # Save immediately
    saved_verdicts.extend([vars(v) for v in new_vs])
    done_queries.add(_q['query'])
    with open(VERDICTS_FILE, 'w') as f:
        json.dump(saved_verdicts, f, indent=2)
    shutil.copy(VERDICTS_FILE, f'{OUTPUT_PATH}/{VERDICTS_FILE}')

    print(f'\nQ3 done. Total verdicts: {len(saved_verdicts)}. Queries done: {len(done_queries)}/{len(eval_queries)}')
    for v in sorted(new_vs, key=lambda x: -x.avg_score):
        print(f'  {{v.system_name[:40]:<40}} avg={{v.avg_score:.2f}}')

In [ ]:
# CELL 10-Q4: Run judge on query 4/8
# This cell skips automatically if already done

_q = eval_queries[3] if 3 < len(eval_queries) else None
if _q is None:
    print('Query 4 not available — fewer than 4 queries generated')
elif _q['query'] in done_queries:
    print(f'Q4 already done: {_q["query"][:60]} — skipping')
else:
    print(f'Q4: {_q["query"]}')
    print(f'  Case: {{_q["case_id"]}} | Topic: {{_q["topic"]}}')
    if _q['majority_hint']:
        print(f'  Majority: {{_q["majority_hint"][:80]}}...')
    if _q['dissent_hint']:
        print(f'  Dissent:  {{_q["dissent_hint"][:80]}}...')
    print()

    new_vs = judge.judge_all(
        systems=systems,
        queries=[_q['query']],
        verbose=True,
    )

    # Save immediately
    saved_verdicts.extend([vars(v) for v in new_vs])
    done_queries.add(_q['query'])
    with open(VERDICTS_FILE, 'w') as f:
        json.dump(saved_verdicts, f, indent=2)
    shutil.copy(VERDICTS_FILE, f'{OUTPUT_PATH}/{VERDICTS_FILE}')

    print(f'\nQ4 done. Total verdicts: {len(saved_verdicts)}. Queries done: {len(done_queries)}/{len(eval_queries)}')
    for v in sorted(new_vs, key=lambda x: -x.avg_score):
        print(f'  {{v.system_name[:40]:<40}} avg={{v.avg_score:.2f}}')

In [ ]:
# CELL 10-Q5: Run judge on query 5/8
# This cell skips automatically if already done

_q = eval_queries[4] if 4 < len(eval_queries) else None
if _q is None:
    print('Query 5 not available — fewer than 5 queries generated')
elif _q['query'] in done_queries:
    print(f'Q5 already done: {_q["query"][:60]} — skipping')
else:
    print(f'Q5: {_q["query"]}')
    print(f'  Case: {{_q["case_id"]}} | Topic: {{_q["topic"]}}')
    if _q['majority_hint']:
        print(f'  Majority: {{_q["majority_hint"][:80]}}...')
    if _q['dissent_hint']:
        print(f'  Dissent:  {{_q["dissent_hint"][:80]}}...')
    print()

    new_vs = judge.judge_all(
        systems=systems,
        queries=[_q['query']],
        verbose=True,
    )

    # Save immediately
    saved_verdicts.extend([vars(v) for v in new_vs])
    done_queries.add(_q['query'])
    with open(VERDICTS_FILE, 'w') as f:
        json.dump(saved_verdicts, f, indent=2)
    shutil.copy(VERDICTS_FILE, f'{OUTPUT_PATH}/{VERDICTS_FILE}')

    print(f'\nQ5 done. Total verdicts: {len(saved_verdicts)}. Queries done: {len(done_queries)}/{len(eval_queries)}')
    for v in sorted(new_vs, key=lambda x: -x.avg_score):
        print(f'  {{v.system_name[:40]:<40}} avg={{v.avg_score:.2f}}')

In [ ]:
# CELL 10-Q6: Run judge on query 6/8
# This cell skips automatically if already done

_q = eval_queries[5] if 5 < len(eval_queries) else None
if _q is None:
    print('Query 6 not available — fewer than 6 queries generated')
elif _q['query'] in done_queries:
    print(f'Q6 already done: {_q["query"][:60]} — skipping')
else:
    print(f'Q6: {_q["query"]}')
    print(f'  Case: {{_q["case_id"]}} | Topic: {{_q["topic"]}}')
    if _q['majority_hint']:
        print(f'  Majority: {{_q["majority_hint"][:80]}}...')
    if _q['dissent_hint']:
        print(f'  Dissent:  {{_q["dissent_hint"][:80]}}...')
    print()

    new_vs = judge.judge_all(
        systems=systems,
        queries=[_q['query']],
        verbose=True,
    )

    # Save immediately
    saved_verdicts.extend([vars(v) for v in new_vs])
    done_queries.add(_q['query'])
    with open(VERDICTS_FILE, 'w') as f:
        json.dump(saved_verdicts, f, indent=2)
    shutil.copy(VERDICTS_FILE, f'{OUTPUT_PATH}/{VERDICTS_FILE}')

    print(f'\nQ6 done. Total verdicts: {len(saved_verdicts)}. Queries done: {len(done_queries)}/{len(eval_queries)}')
    for v in sorted(new_vs, key=lambda x: -x.avg_score):
        print(f'  {{v.system_name[:40]:<40}} avg={{v.avg_score:.2f}}')

In [ ]:
# CELL 10-Q7: Run judge on query 7/8
# This cell skips automatically if already done

_q = eval_queries[6] if 6 < len(eval_queries) else None
if _q is None:
    print('Query 7 not available — fewer than 7 queries generated')
elif _q['query'] in done_queries:
    print(f'Q7 already done: {_q["query"][:60]} — skipping')
else:
    print(f'Q7: {_q["query"]}')
    print(f'  Case: {{_q["case_id"]}} | Topic: {{_q["topic"]}}')
    if _q['majority_hint']:
        print(f'  Majority: {{_q["majority_hint"][:80]}}...')
    if _q['dissent_hint']:
        print(f'  Dissent:  {{_q["dissent_hint"][:80]}}...')
    print()

    new_vs = judge.judge_all(
        systems=systems,
        queries=[_q['query']],
        verbose=True,
    )

    # Save immediately
    saved_verdicts.extend([vars(v) for v in new_vs])
    done_queries.add(_q['query'])
    with open(VERDICTS_FILE, 'w') as f:
        json.dump(saved_verdicts, f, indent=2)
    shutil.copy(VERDICTS_FILE, f'{OUTPUT_PATH}/{VERDICTS_FILE}')

    print(f'\nQ7 done. Total verdicts: {len(saved_verdicts)}. Queries done: {len(done_queries)}/{len(eval_queries)}')
    for v in sorted(new_vs, key=lambda x: -x.avg_score):
        print(f'  {{v.system_name[:40]:<40}} avg={{v.avg_score:.2f}}')

In [ ]:
# CELL 10-Q8: Run judge on query 8/8
# This cell skips automatically if already done

_q = eval_queries[7] if 7 < len(eval_queries) else None
if _q is None:
    print('Query 8 not available — fewer than 8 queries generated')
elif _q['query'] in done_queries:
    print(f'Q8 already done: {_q["query"][:60]} — skipping')
else:
    print(f'Q8: {_q["query"]}')
    print(f'  Case: {{_q["case_id"]}} | Topic: {{_q["topic"]}}')
    if _q['majority_hint']:
        print(f'  Majority: {{_q["majority_hint"][:80]}}...')
    if _q['dissent_hint']:
        print(f'  Dissent:  {{_q["dissent_hint"][:80]}}...')
    print()

    new_vs = judge.judge_all(
        systems=systems,
        queries=[_q['query']],
        verbose=True,
    )

    # Save immediately
    saved_verdicts.extend([vars(v) for v in new_vs])
    done_queries.add(_q['query'])
    with open(VERDICTS_FILE, 'w') as f:
        json.dump(saved_verdicts, f, indent=2)
    shutil.copy(VERDICTS_FILE, f'{OUTPUT_PATH}/{VERDICTS_FILE}')

    print(f'\nQ8 done. Total verdicts: {len(saved_verdicts)}. Queries done: {len(done_queries)}/{len(eval_queries)}')
    for v in sorted(new_vs, key=lambda x: -x.avg_score):
        print(f'  {{v.system_name[:40]:<40}} avg={{v.avg_score:.2f}}')

### Final results — run after all queries are done

In [ ]:
# CELL 10-FINAL: Aggregate all verdicts and generate final plots
import json, shutil
from evaluation_pipeline import (
    JudgeVerdict, aggregate_verdicts, print_ablation_table,
    plot_ablation, save_all_results, evaluate_classification,
)

with open(VERDICTS_FILE) as f:
    raw = json.load(f)

all_verdicts = [JudgeVerdict(**v) for v in raw]
n_done = len(done_queries)
n_total = len(eval_queries)
print(f'Verdicts: {len(all_verdicts)} | Queries: {n_done}/{n_total} done')
if n_done < n_total:
    print(f'WARNING: {n_total - n_done} queries not yet run — results will be partial')

rag_results = aggregate_verdicts(all_verdicts)
clf_results = evaluate_classification(
    test_json='test_split.json',
    bert_model_path='bert_model/best_model',
)

print_ablation_table(rag_results, clf_results)

if rag_results:
    plot_ablation(rag_results, 'ablation_results_8q.png')
    shutil.copy('ablation_results_8q.png', f'{OUTPUT_PATH}/ablation_results_8q.png')

save_all_results(rag_results, clf_results, all_verdicts,
                 'full_evaluation_results_8q.json')
shutil.copy('full_evaluation_results_8q.json',
            f'{OUTPUT_PATH}/full_evaluation_results_8q.json')
print(f'\nSaved to {OUTPUT_PATH}')

## Step 9: Classification Metrics + Error Analysis

In [ ]:
# CELL 11: Classification metrics — structural gold + human gold
import sys, shutil
for mod in list(sys.modules.keys()):
    if 'model_evaluator' in mod: del sys.modules[mod]

from model_evaluator import (
    ModelEvaluator, evaluate_on_human_gold, print_human_vs_structural,
    plot_metric_comparison, plot_confusion_matrices,
    plot_roc_curves, plot_summary_table, save_results,
)

# ── Part A: Structural gold test set (majority/dissent labels) ──
print('=== Part A: Structural Gold Set ===')
ev = ModelEvaluator(
    test_json='test_split.json',
    bert_model_path='bert_model/best_model',
    rule_threshold=0.3,
)
structural_results = ev.evaluate()
ev.print_comparison(structural_results)

for fname, func in [
    ('metric_comparison.png',      plot_metric_comparison),
    ('confusion_matrices.png',     plot_confusion_matrices),
    ('roc_curves.png',             plot_roc_curves),
    ('model_comparison_table.png', plot_summary_table),
]:
    func(structural_results, fname)
    shutil.copy(fname, f'{OUTPUT_PATH}/{fname}')

save_results(structural_results, 'classification_results_structural.json')
shutil.copy('classification_results_structural.json',
            f'{OUTPUT_PATH}/classification_results_structural.json')


In [ ]:
# CELL 11b: Human expert gold set evaluation
# 50 manually annotated pairs, perfect 50/50 balance
# This is the most reliable evaluation — no label noise
import shutil

# Upload human_pairs.json to Colab first:
#   Files panel (left sidebar) → Upload → human_pairs.json
import os
if not os.path.exists('human_pairs.json'):
    print('human_pairs.json not found!')
    print('Upload it via: Files panel → Upload → human_pairs.json')
else:
    print('=== Part B: Human Expert Gold Set ===')
    human_results = evaluate_on_human_gold(
        human_json='human_pairs.json',
        bert_model_path='bert_model/best_model',
        rule_threshold=0.3,
    )

    # Side-by-side comparison: structural vs human labels
    print_human_vs_structural(human_results, structural_results)

    # Separate plots for human gold
    for fname, func in [
        ('metric_comparison_human.png',      plot_metric_comparison),
        ('confusion_matrices_human.png',     plot_confusion_matrices),
        ('roc_curves_human.png',             plot_roc_curves),
        ('model_comparison_table_human.png', plot_summary_table),
    ]:
        func(human_results, fname)
        shutil.copy(fname, f'{OUTPUT_PATH}/{fname}')

    save_results(human_results, 'classification_results_human.json')
    shutil.copy('classification_results_human.json',
                f'{OUTPUT_PATH}/classification_results_human.json')

    print('\n✓ Human gold evaluation complete')
    print(f'  Results saved to {OUTPUT_PATH}')

## Step 10: Graph Visualization

In [ ]:
# CELL 12: Interactive knowledge graph (pyvis)

from pyvis.network import Network
from IPython.display import HTML

net = Network(height='500px', width='100%', bgcolor='#1a1a2e', font_color='white', notebook=True)

party_colors = {
    'prosecution': '#e74c3c', 'defense': '#3498db', 'court': '#2ecc71',
    'majority': '#f39c12',   'dissent': '#9b59b6',  'evidence': '#95a5a6',
}
edge_colors = {
    'CONTRADICTS': '#e74c3c', 'SUPPORTS': '#2ecc71',
    'RESOLVED_BY': '#f39c12', 'HAS_EVIDENCE': '#95a5a6',
}
shown = set()
for nid, data in list(kg.G.nodes(data=True))[:60]:
    color = party_colors.get(data.get('party',''), '#95a5a6')
    net.add_node(nid, label=data.get('text', nid)[:20], color=color,
                 title=data.get('text','')[:200])
    shown.add(nid)
for u, v, data in kg.G.edges(data=True):
    if u in shown and v in shown:
        et = data.get('edge_type','')
        net.add_edge(u, v, label=et, color=edge_colors.get(et,'#555'),
                     width=3 if et=='CONTRADICTS' else 1)


net.save_graph('graph.html')
HTML('graph.html')

---
## Interactive Legal Q&A

**Input:** any question about the cases in the graph

**Output:**
- Structured analytical report: Prosecution → Defense → Court Resolution
- `graph.html` — interactive PyVis subgraph of relevant contradictions

> Run **CELL QA-SETUP** once, then **CELL QA-ASK** as many times as you want.

In [ ]:
# CELL QA-SETUP: Initialize Q&A system
# Run once after the graph is built (Cell 9)
import importlib, sys
if 'legal_qa' in sys.modules:
    importlib.reload(sys.modules['legal_qa'])
from legal_qa import LegalQA

qa = LegalQA(
    kg=kg,
    llm=llm,                        # provider from Cell 2
    top_k=8,                        # nodes to retrieve per query
    graph_output_dir='.',           # where to save graph.html
)
print('Q&A system ready.')
print(f'Graph: {kg.G.number_of_nodes()} nodes, '
      f'{kg.G.number_of_edges()} edges')
print(f'LLM:   {llm.provider} | configured: {llm.is_configured}')

In [ ]:
# CELL QA-ASK: Ask a question — change the text below and re-run
# ─────────────────────────────────────────────────────────────
QUESTION = "Did the defendant knowingly misuse customer funds?"
# ─────────────────────────────────────────────────────────────

result = qa.ask(
    question=QUESTION,
    save_graph=True,
    graph_filename='graph.html',
)

# Print structured report
result.print()

# Copy graph to Drive
import shutil, os
if result.graph_path and os.path.exists(result.graph_path):
    shutil.copy(result.graph_path, f'{OUTPUT_PATH}/graph.html')
    print(f'Graph saved to Drive: {OUTPUT_PATH}/graph.html')

In [ ]:
# CELL QA-GRAPH: Display the interactive graph inline in Colab
result.show_graph()

In [ ]:
# CELL QA-SAVE: Save Q&A result as JSON (for report / presentation)
import json, shutil

output = result.to_dict()
with open('qa_result.json', 'w') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

shutil.copy('qa_result.json', f'{OUTPUT_PATH}/qa_result.json')

print('Saved qa_result.json')
print(f'  Question:    {output["question"]}')
print(f'  Contradictions found: {len(output["contradictions"])}')
print(f'  Graph nodes: {output["n_nodes"]}')
print()
print('── PROSECUTION ──')
print(output['prosecution'][:300])
print('── DEFENSE ──')
print(output['defense'][:300])
print('── COURT ──')
print(output['court'][:200])